In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType

# Clean customers
customers = spark.table("Ecommerce_Project.Bronz_LH.dbo.customers")

customers_clean = (
    customers
    .withColumn("email", lower(trim(col("EMAIL"))))
    .withColumn("name",
      when(trim(col("name")) == "", None).otherwise(col("name")))
    .withColumn("gender", when(lower(col("gender")).isin("f", "female"), "Female")
                          .when(lower(col("gender")).isin("m", "male"), "Male")
                          .otherwise("Other"))
    .withColumn("dob", to_date(regexp_replace(col("dob"), "/", "-")))
    .withColumn("location", initcap(col("location")))
    .dropDuplicates(["customer_id"])
    .dropna(subset=["customer_id", "email"])
)
display(customers_clean)
customers_clean.write.format("delta").mode("overwrite").saveAsTable("Silver_LH.dbo.customers")

StatementMeta(, f6569126-4cd5-4c74-b5db-f6038300cef6, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 755226a0-822d-410e-8969-8dfa2a8f7d6c)

In [26]:
customers_clean.write.format("delta").mode("overwrite").saveAsTable("Ecommerce_Project.Silver_LH.dbo.customers")

StatementMeta(, ab4c28bd-323f-40ae-a95b-916f0442396d, 28, Finished, Available, Finished, False)

In [9]:
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType
# Clean orders
orders = spark.table("Ecommerce_Project.Bronz_LH.dbo.orders")
orders_clean = (
    orders
    .withColumn("order_date", 
        when(col("order_date").rlike("^\d{4}/\d{2}/\d{2}$"), to_date(col("order_date"), "yyyy/MM/dd"))
        .when(col("order_date").rlike("^\d{2}-\d{2}-\d{4}$"), to_date(col("order_date"), "dd-MM-yyyy"))
        .when(col("order_date").rlike("^\d{2}/\d{2}/\d{4}$"), to_date(col("order_date"), "dd/MM/yyyy"))  # ✅ FIX
        .when(col("order_date").rlike("^\d{8}$"), to_date(col("order_date"), "yyyyMMdd"))
        .otherwise(to_date(col("order_date"), "yyyy-MM-dd"))
    )
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .withColumn("status", initcap(col("status")))
    .dropna(subset=["customer_id", "order_date"])
    .dropDuplicates(["order_id"])
)
orders_clean.write.format("delta").mode("overwrite").saveAsTable("Ecommerce_Project.Silver_LH.dbo.orders")
display(orders_clean)

StatementMeta(, 6f126c3b-e8ea-40e1-935f-5c7360decaf4, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bfc13b29-1909-429b-b7bc-2e21b0c4f749)

In [13]:
# Clean payments
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType

# Load table
payments = spark.table("Ecommerce_Project.Bronz_LH.dbo.payments")

# Clean data
payments_clean = (
    payments
    .withColumn(
        "payment_date",
        coalesce(
            to_date(col("payment_date"), "yyyy-MM-dd"),
            to_date(col("payment_date"), "yyyy/MM/dd"),
            to_date(col("payment_date"), "dd-MM-yyyy"),
            to_date(col("payment_date"), "dd/MM/yyyy"),
            to_date(col("payment_date"), "yyyyMMdd")
        )
    )
    .withColumn("payment_method", initcap(trim(col("payment_method"))))
    .replace({"creditcard": "Credit Card"}, subset=["payment_method"])
    .withColumn("payment_status", initcap(trim(col("payment_status"))))
    .withColumn("amount", col("amount").cast("double"))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .dropna(subset=["customer_id", "payment_date", "amount"])
)
# Display result
display(payments_clean)

payments_clean.write.format("delta").mode("overwrite").saveAsTable("Ecommerce_Project.Silver_LH.dbo.payments")

StatementMeta(, 6f126c3b-e8ea-40e1-935f-5c7360decaf4, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 92229401-11af-4406-8a4b-88b54278fdd0)

In [15]:
# Clean support
from pyspark.sql.functions import *

support = spark.table("Ecommerce_Project.Bronz_LH.dbo.support_tickets")

support_clean = (
    support
    # Handle multiple date formats
    .withColumn(
        "ticket_date",
        coalesce(
            to_date(col("ticket_date"), "yyyy-MM-dd"),
            to_date(col("ticket_date"), "yyyy/MM/dd"),
            to_date(col("ticket_date"), "dd-MM-yyyy"),
            to_date(col("ticket_date"), "dd/MM/yyyy"),
            to_date(col("ticket_date"), "yyyyMMdd")
        )
    )

    # Clean text fields
    .withColumn("issue_type", initcap(trim(col("issue_type"))))
    .withColumn("resolution_status", initcap(trim(col("resolution_status"))))

    # Replace invalid values with NULL
    .replace({"NA": None, "": None}, subset=["issue_type", "resolution_status"])

    # Remove duplicates
    .dropDuplicates(["ticket_id"])

    # Remove only critical NULLs
    .dropna(subset=["customer_id", "ticket_date"])
)

display(support_clean)
support_clean.write.format("delta").mode("overwrite").saveAsTable("Ecommerce_Project.Silver_LH.dbo.support_tickets")

StatementMeta(, 6f126c3b-e8ea-40e1-935f-5c7360decaf4, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 62412494-a6f6-4560-82ce-a7ce7c4a2970)

In [16]:
# Clean web

from pyspark.sql.functions import *
web = spark.table("Ecommerce_Project.Bronz_LH.dbo.web_activities")

web_clean = (
    web
    # Handle multiple date formats properly
    .withColumn(
        "session_time",
        coalesce(
            to_date(col("session_time"), "yyyy-MM-dd"),
            to_date(col("session_time"), "yyyy/MM/dd"),
            to_date(col("session_time"), "dd-MM-yyyy"),
            to_date(col("session_time"), "dd/MM/yyyy"),
            to_date(col("session_time"), "yyyyMMdd")
        )
    )

    # Clean text fields
    .withColumn("page_viewed", lower(trim(col("page_viewed"))))
    .withColumn("device_type", initcap(trim(col("device_type"))))

    # Replace invalid values
    .replace({"": None}, subset=["page_viewed"])

    # Remove duplicates
    .dropDuplicates(["session_id"])

    # Remove only critical nulls
    .dropna(subset=["customer_id", "session_time", "page_viewed"])
)

display(web_clean)
web_clean.write.format("delta").mode("overwrite").saveAsTable("Ecommerce_Project.Silver_LH.dbo.web_activities")

StatementMeta(, 6f126c3b-e8ea-40e1-935f-5c7360decaf4, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5546940a-7e55-43a0-892c-9377dc1b8f03)